In [1]:
!nvidia-smi
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else 'N/A')

Thu Jul  9 04:36:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   32C    P0             83W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install -qU peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 68.3 MB/s eta 0:00:00


In [2]:
import os
from google.colab import files

if not os.path.exists('dataset.jsonl'):
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.jsonl'):
            os.rename(fn, 'dataset.jsonl')
            print(f'Uploaded {fn}')
else:
    print('dataset.jsonl already present')

dataset.jsonl already present


In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
# Configuration
import os
MODEL_CHOICE = "qwen"  # or "qwen"

# HuggingFace token (required for Llama)
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN') or ""
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

if MODEL_CHOICE == "llama":
    BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
    OUTPUT_DIR = "lora_llama"
    BATCH_SIZE = 1
    GRAD_ACCUM = 4
else:
    BASE_MODEL = "Qwen/Qwen3-30B-A3B"
    OUTPUT_DIR = "lora_qwen"
    BATCH_SIZE = 1
    GRAD_ACCUM = 8

Now to load dataset and train

In [5]:
import json, os
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [9]:
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
splits = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = splits["train"]
eval_dataset = splits["test"]
print(f"Dataset: {len(train_dataset)} examples {len(eval_dataset)} eval")

Dataset: 251 examples 28 eval


In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    num_train_epochs=2,
    learning_rate=1e-4,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_total_limit=1,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to="none",
    optim="adamw_8bit",
    eval_strategy="steps", eval_steps=10, load_best_model_at_end=True, metric_for_best_model="loss"
)

In [14]:
with torch.no_grad():
  torch.cuda.empty_cache()

In [15]:
with torch.no_grad():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading {BASE_MODEL}...")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
    )
    # Avoid the OOM upcast step entirely by using manual configuration
    model.enable_input_require_grads()
    model.config.use_cache = False

    peft_config = LoraConfig(
        r=8, lora_alpha=16, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

Loading Qwen/Qwen3-30B-A3B...


Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 497,418,240 || all params: 31,029,540,864 || trainable%: 1.6030


In [16]:
def format_chat(example):
    kwargs = dict(tokenize=False, add_generation_prompt=False)
    if MODEL_CHOICE == "qwen":
        kwargs["enable_thinking"] = False
    return tokenizer.apply_chat_template(example["messages"], **kwargs)


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=format_chat
)

trainer.train()


Applying formatting function to train dataset:   0%|          | 0/251 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/251 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/251 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/28 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/28 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/28 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss


In [16]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('lora_qwen/tokenizer_config.json',
 'lora_qwen/chat_template.jinja',
 'lora_qwen/tokenizer.json')

In [ ]:
cfg_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
if os.path.exists(cfg_path):
    with open(cfg_path) as f:
        cfg = json.load(f)
    print(f"Model type: {cfg.get('model_type', 'NOT SET')}")